# E046 — Atlas visuel complet

Cet atlas affiche les images plutôt que de résumer uniquement les scores :

- Stage 1 brut / scene-preserving ;
- Stage 2 brut / scene-preserving ;
- chaque trajectoire SR-MPGD, toutes les itérations ;
- différences périphériques et différences dans le cœur ;
- meilleurs, échecs, no-op et Pareto.

La bordure uniforme blanche/adaptive-light n'existe pas dans les sorties E046.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display, Markdown, Image

OUTPUT_ROOT = Path(os.environ.get(
    "PROOFTAG_E046_OUTPUT_ROOT",
    "/data/e046-controlled-best-generator-v1",
))
latest = json.loads((OUTPUT_ROOT / "LATEST.json").read_text(encoding="utf-8"))
R = Path(latest["plan_dir"])
plan = json.loads((R / "plan.json").read_text(encoding="utf-8"))
verdict_path = R / "verdict.json"
complete_path = R / "COMPLETE.json"
verdict = (
    json.loads(verdict_path.read_text(encoding="utf-8"))
    if verdict_path.is_file()
    else None
)

def load_rows():
    final = R / "dataset/e046-observations.json"
    if final.is_file():
        return json.loads(final.read_text(encoding="utf-8"))

    rows = []
    for path in sorted((R / "parents").glob("*/scoring/comparison.json")):
        rows.extend(json.loads(path.read_text(encoding="utf-8")))
    for path in sorted((R / "refinements").glob("*/*/scoring/comparison.json")):
        rows.extend(json.loads(path.read_text(encoding="utf-8")))
    return rows

rows = load_rows()
df = pd.DataFrame(rows)
for column, default in (
    ("srmpgd_recipe_id", None),
    ("row_id", None),
    ("pixel_duplicate", False),
    ("projection_was_active", False),
):
    if column not in df.columns:
        df[column] = default
print("Plan E046 :", R)
print("Profile   :", plan["profile"])
print("Status    :", latest.get("status"))
print("Rows      :", len(df))
print("Engine QR :", plan["qr_software_engine"])

In [ ]:
if df.empty:
    raise RuntimeError("Aucune image E046 scorée. Ouvrir plus tard ou utiliser le notebook principal.")
for column in ("wechat_exact_presets", "clip_aesthetic", "hpsv2_1", "lpips"):
    df[column] = pd.to_numeric(df[column], errors="coerce")

## 1. Stage 1 / Stage 2 pour chaque parent

In [ ]:
for candidate_id, group in df[df["source_kind"] == "parent"].groupby("candidate_id"):
    order = ["stage1_raw", "stage1_scene_qz", "stage2_raw", "stage2_scene_qz"]
    records = {
        row["variant"]: row
        for _, row in group.iterrows()
    }
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for axis, variant in zip(axes, order):
        row = records.get(variant)
        if row is not None and Path(str(row["image_path"])).is_file():
            axis.imshow(PILImage.open(row["image_path"]).convert("RGB"))
            axis.set_title(
                f"{variant}\nWeChat {int(row['wechat_exact_presets'])}/37\n"
                f"AES {float(row['clip_aesthetic']):.3f}"
            )
        else:
            axis.set_title(f"{variant}\nabsent")
        axis.axis("off")
    plt.suptitle(candidate_id)
    plt.tight_layout()
    plt.show()

## 2. Différence brut → scene-preserving

In [ ]:
def diff_map(left_path, right_path):
    left = np.asarray(PILImage.open(left_path).convert("RGB"), dtype=np.float32) / 255
    right = np.asarray(PILImage.open(right_path).convert("RGB"), dtype=np.float32) / 255
    return np.abs(right - left).mean(axis=2)

for candidate_id, group in df[df["source_kind"] == "parent"].groupby("candidate_id"):
    records = {row["variant"]: row for _, row in group.iterrows()}
    if "stage2_raw" not in records or "stage2_scene_qz" not in records:
        continue
    raw = records["stage2_raw"]
    scene = records["stage2_scene_qz"]
    delta = diff_map(raw["image_path"], scene["image_path"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(PILImage.open(raw["image_path"]).convert("RGB"))
    axes[0].set_title("Stage 2 brut")
    axes[1].imshow(PILImage.open(scene["image_path"]).convert("RGB"))
    axes[1].set_title("Scene-preserving")
    im = axes[2].imshow(delta)
    axes[2].set_title("Différence absolue moyenne")
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    for axis in axes:
        axis.axis("off")
    plt.suptitle(
        f"{candidate_id} — cœur inchangé: {scene['core_byte_identical_to_raw']}"
    )
    plt.tight_layout()
    plt.show()

## 3. Toutes les trajectoires SR-MPGD

In [ ]:
sr = df[df["source_kind"] == "srmpgd"]
if sr.empty:
    display(Markdown("Aucune trajectoire scorée."))
else:
    for (candidate_id, recipe_id, qz_variant), group in sr.groupby(
        ["candidate_id", "srmpgd_recipe_id", "quiet_zone_variant"]
    ):
        group = group.sort_values("iteration")
        columns = 5
        nrows = math.ceil(len(group) / columns)
        fig, axes = plt.subplots(nrows, columns, figsize=(18, 4.2 * nrows))
        axes = np.asarray(axes).reshape(-1)
        for axis in axes:
            axis.axis("off")
        for axis, (_, row) in zip(axes, group.iterrows()):
            path = Path(str(row["image_path"]))
            if path.is_file():
                axis.imshow(PILImage.open(path).convert("RGB"))
            axis.set_title(
                f"i{int(row['iteration'])}\n"
                f"WeChat {int(row['wechat_exact_presets'])}/37\n"
                f"LPIPS {float(row['lpips']):.4f}",
                fontsize=9,
            )
            axis.axis("off")
        plt.suptitle(f"{candidate_id} — {recipe_id} — {qz_variant}")
        plt.tight_layout()
        plt.show()

## 4. Courbes image + métriques pour chaque trajectoire

In [ ]:
if not sr.empty:
    raw = sr[sr["quiet_zone_variant"] == "raw"]
    for (candidate_id, recipe_id), group in raw.groupby(
        ["candidate_id", "srmpgd_recipe_id"]
    ):
        group = group.sort_values("iteration")
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax1.plot(
            group["iteration"],
            group["wechat_exact_presets"],
            marker="o",
            label="WeChat exact /37",
        )
        ax1.set_xlabel("Itération")
        ax1.set_ylabel("WeChat exact /37")
        ax1.set_ylim(-0.5, 37.5)
        ax2 = ax1.twinx()
        ax2.plot(
            group["iteration"],
            group["lpips"],
            marker="x",
            label="LPIPS",
        )
        ax2.set_ylabel("LPIPS")
        plt.title(f"{candidate_id} — {recipe_id}")
        fig.tight_layout()
        plt.show()

## 5. Top logiciel sous garde visuelle

In [ ]:
safe = df[df["eligible_final"] == True].sort_values(
    ["wechat_exact_presets", "clip_aesthetic", "hpsv2_1"],
    ascending=[False, False, False],
)
top = safe.head(32).to_dict("records")

if not top:
    display(Markdown("Aucun candidat ne passe encore la garde visuelle."))
else:
    columns = 4
    nrows = math.ceil(len(top) / columns)
    fig, axes = plt.subplots(nrows, columns, figsize=(16, 4.6 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row in zip(axes, top):
        path = Path(str(row["image_path"]))
        if path.is_file():
            axis.imshow(PILImage.open(path).convert("RGB"))
        axis.set_title(
            f"{row['candidate_id']}\n"
            f"{row['source_kind']} {row['variant']}\n"
            f"WeChat {int(row['wechat_exact_presets'])}/37 · "
            f"AES {float(row['clip_aesthetic']):.2f}",
            fontsize=8,
        )
        axis.axis("off")
    plt.suptitle("Top E046 — logiciel et garde visuelle")
    plt.tight_layout()
    plt.show()

## 6. Meilleurs esthétiques parmi les faibles scores QR

In [ ]:
low = (
    df[(df["eligible_final"] == True) & (df["wechat_exact_presets"] <= 5)]
    .sort_values(["clip_aesthetic", "hpsv2_1"], ascending=False)
    .head(24)
)
if low.empty:
    display(Markdown("Aucun exemple faible WeChat sous garde."))
else:
    columns = 4
    nrows = math.ceil(len(low) / columns)
    fig, axes = plt.subplots(nrows, columns, figsize=(16, 4.5 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, (_, row) in zip(axes, low.iterrows()):
        axis.imshow(PILImage.open(row["image_path"]).convert("RGB"))
        axis.set_title(
            f"{row['prompt_id']}\nWeChat {int(row['wechat_exact_presets'])}/37\n"
            f"AES {float(row['clip_aesthetic']):.2f}",
            fontsize=9,
        )
        axis.axis("off")
    plt.suptitle("Hard negatives esthétiques")
    plt.tight_layout()
    plt.show()

## 7. Pareto complet

In [ ]:
pareto_path = R / "dataset/pareto-front.json"
if pareto_path.is_file():
    pareto = json.loads(pareto_path.read_text(encoding="utf-8"))
    columns = 4
    nrows = math.ceil(len(pareto) / columns)
    fig, axes = plt.subplots(nrows, columns, figsize=(16, 4.6 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row in zip(axes, pareto):
        axis.imshow(PILImage.open(row["image_path"]).convert("RGB"))
        axis.set_title(
            f"{row['candidate_id']}\n{row['variant']}\n"
            f"WeChat {row['wechat_exact_presets']}/37 · "
            f"AES {float(row['clip_aesthetic']):.2f}",
            fontsize=8,
        )
        axis.axis("off")
    plt.suptitle("Front de Pareto E046")
    plt.tight_layout()
    plt.show()

## 8. Gagnant final et rappel de sécurité

In [ ]:
if verdict is not None:
    final_path = R / "pipeline/99-FINAL-QR.png"
    display(Image(filename=str(final_path)))
    display({
        "winner": verdict["winner_candidate_id"],
        "variant": verdict["winner_variant"],
        "WeChat_exact": f"{verdict['winner_wechat_exact_presets']}/37",
        "original_exact": verdict["winner_wechat_original_exact"],
        "uniform_quiet_zone_replacement": verdict[
            "winner_uniform_quiet_zone_replacement"
        ],
        "phone_validated": verdict["phone_truth_available"],
        "production_ready": verdict["production_ready"],
    })

Aucun gagnant E046 ne doit être appelé « fonctionnel téléphone » avant la campagne
physique. L'atlas sert précisément à choisir les images à tester sur appareils.